# Candidate Recovery Metric — Sleep Duration and Efficiency

Sleep is the wearable signal patients themselves most associate with recovery, and it is recorded
passively overnight rather than requiring the patient to move. Against that, **it is the signal
most easily disrupted by things that are not recovery** — a hospital ward, analgesia, a device
removed at night — so its face validity as a recovery measure is weaker than its coverage
suggests. This notebook audits whether sleep is measured densely enough around surgery to support
an outcome, whether the signal moves at all, and whether any movement it shows survives being
measured within patient rather than across the cohort. Efficiency is carried alongside duration
because the two can disagree, and disagreement would mean two candidate metrics rather than one.

**Governance.** Every query aggregates inside BigQuery and returns counts, quantiles or
correlations only, so no participant-level rows enter the notebook. Unsuppressed figures stay
git-ignored while `*_suppressed.png` is committable, and the small-cell threshold must be
confirmed against current All of Us policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored figure
and cache directories, and builds the cohort plus one index date per patient. **The metric CTE
and both reference panels are defined here**, so every SQL definition the notebook runs on sits
in one place. `is_main_sleep` is a STRING in this CDR rather than a BOOL, so it is cast before
comparison — assume nothing about a column's type, look it up with `srp_common.column_type`.

#### Imports, Cohort SQL, the Sleep CTE and the Reference Panels

Which of a patient's surgery-days becomes the index date is still open, so `INDEX_RULE` is
`"first"` — the only rule no post-operative event can influence — and every result below is
stamped with it. **The two reference panels are separate on purpose**: efficiency needs minutes
in bed as well as minutes asleep, so a patient can clear seven nights of recorded sleep while
carrying only two usable efficiency values, and a median of two is not a baseline.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_sleep")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = f"""
    sleep AS (
        SELECT
            s.person_id,
            DATE_DIFF(s.sleep_date, i.index_date, DAY) AS rel_day,
            s.minute_in_bed,
            s.minute_asleep,
            SAFE_DIVIDE(s.minute_asleep, NULLIF(s.minute_in_bed, 0)) AS efficiency
        FROM `{CDR}`.sleep_daily_summary AS s
        JOIN index_surgery AS i ON i.person_id = s.person_id
        -- is_main_sleep is a STRING ('true'/'false') in this CDR, not a BOOL, so a
        -- bare `= true` is a type error. Casting first works either way, and covers
        -- a 1/0 encoding too, so a CDR release that changes the type does not break
        -- this query.
        WHERE LOWER(CAST(s.is_main_sleep AS STRING)) IN ('true', '1')
          AND s.minute_asleep IS NOT NULL
    )"""

# Every day-by-day median below is taken over whoever recorded a night, and that
# set is not fixed -- so a fall in the curve can be a change in who is contributing
# rather than a change in sleep. PANEL_CTE holds the contributors fixed: patients
# with a settled sleep habit well before surgery, each carrying their own reference
# median to difference against. The band ends at day -61 so it does not overlap the
# run-up to surgery that the panel exists to examine.
PANEL_BAND = (-180, -61)
PANEL_MIN_NIGHTS = 7

PANEL_CTE = f"""
    , panel AS (
        SELECT
            person_id,
            APPROX_QUANTILES(minute_asleep, 100)[OFFSET(50)] AS ref_minutes
        FROM sleep
        WHERE rel_day BETWEEN {PANEL_BAND[0]} AND {PANEL_BAND[1]}
        GROUP BY person_id
        HAVING COUNT(DISTINCT rel_day) >= {PANEL_MIN_NIGHTS}
    )
"""

# Efficiency gets its own reference panel rather than another column on PANEL_CTE.
# It needs minutes in bed as well as minutes asleep, so a patient can clear seven
# nights of recorded sleep while carrying only two usable efficiency values, and a
# median of two is not a baseline. Same band and same minimum, so the two panels are
# defined alike and can be read beside each other.
EFFICIENCY_PANEL_CTE = f"""
    , efficiency_panel AS (
        SELECT
            person_id,
            APPROX_QUANTILES(efficiency, 100)[OFFSET(50)] AS ref_efficiency
        FROM sleep
        WHERE rel_day BETWEEN {PANEL_BAND[0]} AND {PANEL_BAND[1]}
          AND efficiency IS NOT NULL
        GROUP BY person_id
        HAVING COUNT(DISTINCT rel_day) >= {PANEL_MIN_NIGHTS}
    )
"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "sleep_daily_summary": ["person_id", "sleep_date", "is_main_sleep", "minute_in_bed", "minute_asleep"],
})

## Part 1 — Is Sleep Measured Densely Enough

Coverage first, because no amount of signal rescues a metric the cohort does not carry. Naps are
excluded, so this counts nights rather than sleep episodes. **The peri-operative dip is the part
to read: a device left at home during an admission produces exactly the pattern a recovery metric
cannot survive.** The paired test is the same one the other wearable metrics use, so the three
stay directly comparable in the scorecard, and seven nights is the minimum that makes a
per-patient baseline more than noise.

#### Any-Record Coverage and the Paired Windows

`sleep_daily_summary` carries one row per sleep episode and the `is_main_sleep` flag is what
separates a night's sleep from a nap, so naps are already out before any count is taken. **A night
with no record contributes nothing rather than a zero**, which is why the denominator moves from
day to day and why Part 3 exists at all.

In [ ]:
# Any-record coverage plus the curve against distance from surgery. Naps are already
# excluded by the CTE, so this counts NIGHTS rather than sleep episodes.
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                    AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM sleep)    AS patients_with_any_sleep,
        (SELECT COUNT(*) FROM sleep)                     AS sleep_nights
"""

cov = cached_gbq(coverage_sql, "sleep_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients             : {cohort_patients:>9,}")
print(f"  ... with >=1 main sleep   : {int(cov['patients_with_any_sleep']):>9,}"
      f"   ({int(cov['patients_with_any_sleep']) / cohort_patients:6.1%} of cohort)")
print(f"main-sleep nights           : {int(cov['sleep_nights']):>9,}")

curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM sleep
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

sleep_curve = cached_gbq(curve_sql, "sleep_coverage_curve")

srp_common.plot_rel_day_curve(
    cfg, sleep_curve, "n_patients",
    title="Patients with a Fitbit main-sleep record, by day relative to surgery",
    ylabel="patients with a sleep record", suppress=False, save_as="sleep_coverage")
srp_common.plot_rel_day_curve(
    cfg, sleep_curve, "n_patients",
    title="Patients with a Fitbit main-sleep record, by day relative to surgery",
    ylabel="patients with a sleep record", suppress=True, save_as="sleep_coverage")

# The paired test, identical to the one steps and heart rate use, so the three
# wearable metrics stay comparable in the scorecard.
window_case = srp_common.window_case_sql("rel_day")

paired_sql = CTES + f"""
    , per_window AS (
        SELECT
            person_id,
            {window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM sleep
        WHERE {window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    wide AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'postop',   n_days, 0)) AS postop_days
        FROM per_window
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                            AS patients_any_window,
        COUNTIF(baseline_days >= 7)                         AS baseline_7d,
        COUNTIF(postop_days >= 7)                           AS postop_7d,
        COUNTIF(baseline_days >= 7 AND postop_days >= 7)    AS paired_7d
    FROM wide
"""

paired = cached_gbq(paired_sql, "sleep_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "postop_7d", "paired_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

## Part 2 — Does the Signal Move

Duration and efficiency are plotted on separate axes rather than one, because they are different
measures on different scales and a shared axis would misstate both. **If only one of them
responds to surgery, only that one is a candidate.** Everything in this Part is cross-sectional —
each day's median is over whoever recorded a night that day — which is the limitation Part 3
exists to test.

#### The Peri-Operative Curves for Duration and Efficiency

Quantiles are computed inside BigQuery so only the per-day summary crosses into the notebook.
**The IQR band is the spread across patients on that day, not a confidence interval** — it does
not narrow as more patients contribute, and a narrow band means patients resemble each other
rather than that the median is precisely estimated.

In [ ]:
signal_sql = CTES + """
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                              AS n_patients,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(50)]       AS p50_minutes_asleep,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(25)]       AS p25_minutes_asleep,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(75)]       AS p75_minutes_asleep,
        APPROX_QUANTILES(efficiency, 100)[OFFSET(50)]          AS p50_efficiency
    FROM sleep
    WHERE rel_day BETWEEN -60 AND 180
      AND efficiency IS NOT NULL
    GROUP BY rel_day
    ORDER BY rel_day
"""

sleep_signal = cached_gbq(signal_sql, "sleep_signal_curve")

srp_common.plot_rel_day_curve(
    cfg, sleep_signal, "p50_minutes_asleep",
    band=("p25_minutes_asleep", "p75_minutes_asleep"),
    title="Median minutes asleep (IQR band), by day relative to surgery",
    ylabel="minutes asleep", suppress=True, save_as="sleep_duration")
srp_common.plot_rel_day_curve(
    cfg, sleep_signal, "p50_efficiency",
    title="Median sleep efficiency, by day relative to surgery",
    ylabel="minutes asleep / minutes in bed", suppress=True, save_as="sleep_efficiency")

base_lo, base_hi = WINDOWS["baseline"]
for column, label in [("p50_minutes_asleep", "minutes asleep"),
                      ("p50_efficiency", "efficiency")]:
    baseline = sleep_signal.loc[
        sleep_signal["rel_day"].between(base_lo, base_hi), column].median()
    postop = sleep_signal.loc[
        sleep_signal["rel_day"].between(*WINDOWS["postop"]), column].median()
    print(f"{label:16s} baseline={baseline:8.3f}  postop={postop:8.3f}  "
          f"change={postop - baseline:+8.3f}  ({(postop - baseline) / baseline:+6.1%})")

## Part 3 — Is Any of It Just People Not Wearing the Device

Every median above is cross-sectional: each one is taken over whoever recorded a night that day,
and that set shrinks toward surgery. **A decline is therefore consistent with two different
stories — patients sleeping less, or the patients who sleep longest being the ones who stop
wearing the device — and Part 2 cannot tell them apart.** Everything here runs against a fixed
panel defined in setup, patients with at least seven main-sleep nights between 180 and 61 days
before surgery, each carrying their own reference median. The band stops at day -61 so it does
not overlap the run-up being examined, and days -90 to -61 overlap the reference band, so a
within-patient change of about zero there is the arithmetic check that the anchoring works.

#### Adherence, the Within-Patient Change, and Time in Bed

Three readings of the same fixed panel. **A flat within-patient curve under a falling
cross-sectional one means the decline was who was wearing the device, not how anyone slept**;
minutes in bed falling in step with minutes asleep while efficiency holds means the night was
truncated by a device coming off rather than genuinely shorter.

In [ ]:
wear_sql = CTES + PANEL_CTE + """
    SELECT
        s.rel_day,
        COUNT(DISTINCT s.person_id)                                 AS n_patients,
        (SELECT COUNT(*) FROM panel)                                AS n_panel,
        COUNT(DISTINCT s.person_id) / (SELECT COUNT(*) FROM panel)  AS wear_share
    FROM sleep AS s
    JOIN panel AS p ON p.person_id = s.person_id
    WHERE s.rel_day BETWEEN -90 AND 90
    GROUP BY s.rel_day
    ORDER BY s.rel_day
"""

wear = cached_gbq(wear_sql, "sleep_panel_wear")
n_panel = int(wear["n_panel"].iloc[0])

print(f"panel: {n_panel:,} patients with >={PANEL_MIN_NIGHTS} nights in "
      f"[{PANEL_BAND[0]}, {PANEL_BAND[1]}]   ({n_panel / cohort_patients:.1%} of cohort)")

srp_common.plot_rel_day_curve(
    cfg, wear, "wear_share",
    title="Share of a fixed panel recording a main sleep, by day relative to surgery",
    ylabel="share of the panel with a sleep record", suppress=True, save_as="panel_wear")

for day in [-60, -30, -14, -7, -1, 1, 7, 30]:
    row = wear.loc[wear["rel_day"] == day]
    if row.empty or int(row["n_patients"].iloc[0]) < SUPPRESS_THRESHOLD:
        print(f"day {day:+4d}   below the {SUPPRESS_THRESHOLD}-patient threshold, not reported")
    else:
        print(f"day {day:+4d}   {row['wear_share'].iloc[0]:6.1%} of the panel recorded a night")

# The within-patient change. Differencing against each patient's OWN reference removes
# composition by construction: a patient enters a day's median only by contributing
# to it and having a reference.
delta_sql = CTES + PANEL_CTE + """
    SELECT
        s.rel_day,
        COUNT(DISTINCT s.person_id)                                        AS n_patients,
        APPROX_QUANTILES(s.minute_asleep, 100)[OFFSET(50)]                 AS p50_minutes_asleep,
        APPROX_QUANTILES(s.minute_asleep - p.ref_minutes, 100)[OFFSET(50)] AS p50_delta,
        APPROX_QUANTILES(s.minute_asleep - p.ref_minutes, 100)[OFFSET(25)] AS p25_delta,
        APPROX_QUANTILES(s.minute_asleep - p.ref_minutes, 100)[OFFSET(75)] AS p75_delta
    FROM sleep AS s
    JOIN panel AS p ON p.person_id = s.person_id
    WHERE s.rel_day BETWEEN -90 AND 90
    GROUP BY s.rel_day
    ORDER BY s.rel_day
"""

panel_signal = cached_gbq(delta_sql, "sleep_panel_delta")

srp_common.plot_rel_day_curve(
    cfg, panel_signal, "p50_delta", band=("p25_delta", "p75_delta"),
    title="Median within-patient change in minutes asleep (IQR band), by day relative to surgery",
    ylabel="minutes asleep - that patient's reference median",
    suppress=True, zero_line=True, save_as="panel_delta")

# Days -90..-61 overlap the reference band, so a within-patient change of about zero
# there is the arithmetic sanity check, not a result.
readable = panel_signal.loc[panel_signal["n_patients"] >= SUPPRESS_THRESHOLD]
for lo, hi, label in [(-90, -61, "reference band"), (-30, -1, "baseline window"),
                      (-7, -1, "week before"), (1, 30, "postop window")]:
    seg = readable.loc[readable["rel_day"].between(lo, hi)]
    if seg.empty:
        print(f"{label:16s} nothing above the {SUPPRESS_THRESHOLD}-patient threshold")
        continue
    print(f"{label:16s} cross-sectional median={seg['p50_minutes_asleep'].median():7.1f} min"
          f"   within-patient change={seg['p50_delta'].median():+7.1f} min")

# Minutes in bed and efficiency for the same panel. A device taken off mid-night
# truncates in-bed and asleep TOGETHER while leaving their ratio alone.
in_bed_sql = CTES + PANEL_CTE + """
    SELECT
        s.rel_day,
        -- minute_in_bed is nullable where minute_asleep is not, so this counts the
        -- patients who actually contribute an in-bed value and SAFE_OFFSET covers a
        -- day on which none of them do.
        COUNT(DISTINCT IF(s.minute_in_bed IS NOT NULL, s.person_id, NULL)) AS n_patients,
        APPROX_QUANTILES(s.minute_in_bed, 100)[SAFE_OFFSET(50)]            AS p50_minute_in_bed,
        APPROX_QUANTILES(s.efficiency, 100)[SAFE_OFFSET(50)]               AS p50_efficiency
    FROM sleep AS s
    JOIN panel AS p ON p.person_id = s.person_id
    WHERE s.rel_day BETWEEN -90 AND 90
    GROUP BY s.rel_day
    ORDER BY s.rel_day
"""

in_bed = cached_gbq(in_bed_sql, "sleep_panel_in_bed")

srp_common.plot_rel_day_curve(
    cfg, in_bed, "p50_minute_in_bed",
    title="Median minutes in bed for the same panel, by day relative to surgery",
    ylabel="minutes in bed", suppress=True, save_as="panel_in_bed")

both = panel_signal.merge(in_bed, on="rel_day", suffixes=("_asleep", "_in_bed"))
both = both.loc[(both["n_patients_asleep"] >= SUPPRESS_THRESHOLD)
                & (both["n_patients_in_bed"] >= SUPPRESS_THRESHOLD)]
ref = both.loc[both["rel_day"].between(*PANEL_BAND)]

for lo, hi, label in [(-30, -1, "baseline window"), (-7, -1, "week before"),
                      (1, 30, "postop window")]:
    seg = both.loc[both["rel_day"].between(lo, hi)]
    if seg.empty or ref.empty:
        print(f"{label:16s} nothing above the {SUPPRESS_THRESHOLD}-patient threshold")
        continue
    print(f"{label:16s} asleep {seg['p50_minutes_asleep'].median() - ref['p50_minutes_asleep'].median():+7.1f} min"
          f"   in bed {seg['p50_minute_in_bed'].median() - ref['p50_minute_in_bed'].median():+7.1f} min"
          f"   efficiency {seg['p50_efficiency'].median() - ref['p50_efficiency'].median():+7.3f}")

#### Efficiency Up Close, Cross-Sectionally and Within Patient

The wider efficiency curve spends most of its width on days that are not in question, so a dip
confined to the admission is a few pixels there and invisible; the first zoom is a slice of the
already-computed frame rather than a second query, so it cannot disagree with the view it was cut
from. The second differences every night against that patient's own reference, and **zero on that
plot is the night the patient usually has** — a band entirely below zero means at least three
quarters of patients were worse than their own baseline, which is far stronger than a median
line.

In [ ]:
# CROSS-SECTIONAL zoom: a slice of sleep_signal, not a new query. n_patients here is
# the efficiency denominator specifically, since that query filtered on
# efficiency IS NOT NULL before counting.
# A slice of the curve already computed above rather than a second query: sleep_signal
# spans -60..180 and the same rows produce both plots, so the zoom cannot disagree with
# the wider view it was cut from. n_patients is the efficiency denominator specifically --
# that query filters on efficiency IS NOT NULL before counting, so nights carrying sleep
# but no time in bed are already out of both the median and the count.
efficiency_zoom = sleep_signal.loc[sleep_signal["rel_day"].between(-10, 10)]

# show_windows=False because the baseline and postop spans are wider than this view:
# their shading would cover it end to end and their labels would land off-axis.
srp_common.plot_rel_day_curve(
    cfg, efficiency_zoom, "p50_efficiency",
    title="Median sleep efficiency, ten days either side of surgery",
    ylabel="minutes asleep / minutes in bed", suppress=True,
    show_windows=False, save_as="sleep_efficiency_zoom")

for _, row in efficiency_zoom.iterrows():
    day, n = int(row["rel_day"]), int(row["n_patients"])
    if n < SUPPRESS_THRESHOLD:
        print(f"day {day:+3d}   below the {SUPPRESS_THRESHOLD}-patient threshold, not reported")
    else:
        print(f"day {day:+3d}   efficiency={row['p50_efficiency']:.3f}   "
              f"patients contributing={n:,}")

# WITHIN-PATIENT zoom over the same days, against each patient's own reference.
efficiency_delta_sql = CTES + EFFICIENCY_PANEL_CTE + """
    SELECT
        s.rel_day,
        COUNT(DISTINCT s.person_id)                                        AS n_patients,
        APPROX_QUANTILES(s.efficiency, 100)[OFFSET(50)]                    AS p50_efficiency,
        APPROX_QUANTILES(s.efficiency - e.ref_efficiency, 100)[OFFSET(50)] AS p50_delta,
        APPROX_QUANTILES(s.efficiency - e.ref_efficiency, 100)[OFFSET(25)] AS p25_delta,
        APPROX_QUANTILES(s.efficiency - e.ref_efficiency, 100)[OFFSET(75)] AS p75_delta
    FROM sleep AS s
    JOIN efficiency_panel AS e ON e.person_id = s.person_id
    -- ref_efficiency is non-null by construction and this filter makes s.efficiency
    -- non-null too, so every surviving day has at least one value and OFFSET cannot
    -- index into a NULL quantile array.
    WHERE s.rel_day BETWEEN -10 AND 10
      AND s.efficiency IS NOT NULL
    GROUP BY s.rel_day
    ORDER BY s.rel_day
"""

efficiency_delta = cached_gbq(efficiency_delta_sql, "sleep_efficiency_delta_zoom")

srp_common.plot_rel_day_curve(
    cfg, efficiency_delta, "p50_delta", band=("p25_delta", "p75_delta"),
    title="Median within-patient change in sleep efficiency, ten days either side of surgery",
    ylabel="efficiency - that patient's reference efficiency",
    suppress=True, show_windows=False, zero_line=True,
    save_as="sleep_efficiency_delta_zoom")

for _, row in efficiency_delta.iterrows():
    day, n = int(row["rel_day"]), int(row["n_patients"])
    if n < SUPPRESS_THRESHOLD:
        print(f"day {day:+3d}   below the {SUPPRESS_THRESHOLD}-patient threshold, not reported")
    else:
        print(f"day {day:+3d}   change={row['p50_delta']:+.3f}"
              f"   cross-sectional={row['p50_efficiency']:.3f}"
              f"   patients contributing={n:,}")

## Part 4 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Face validity is the weak point for this metric rather than coverage, and that judgement is
recorded in the row's notes so a later reader can disagree with it specifically. Ward noise,
analgesia and a device left off overnight all move sleep without being recovery.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="sleep_daily_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_sleep"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Face validity is the weak point, not coverage: ward noise, analgesia and a device "
           "left off overnight all move sleep without being recovery."),
)